# Configuration

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

%load_ext autoreload
%autoreload 2

/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/notebooks
/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/app


# Extract

In [ ]:
from transformers import pipeline
import librosa

# Setup device: 0 for Nvidia GPU, -1 for CPU
hf_device = 0# if device == "cuda" else -1 

# Loads HuBERT for tone/emotion (neutral, happy, angry, sad)
emotion_model = pipeline("audio-classification", model="superb/hubert-large-superb-er", device=hf_device)

# Loads AudioSpectrogramTransformer for 527 background sounds (music, laughter, sirens, etc.)
event_model = pipeline("audio-classification", model="MIT/ast-finetuned-audioset-10-10-0.4593", device=hf_device)

In [ ]:
def extract_audio_cues(video_path):
    # These models strictly require 16kHz audio format
    audio, _ = librosa.load(video_path, sr=16000)
    
    # Extract the dominant tone
    emotion_result = emotion_model(audio, top_k=1)
    tone = emotion_result[0]['label']
    
    # Extract the top 3 background sounds/events
    event_result = event_model(audio, top_k=3)
    background_sounds = [event['label'] for event in event_result]
    
    return tone, background_sounds


In [ ]:
from maikol_utils.file_utils import load_json
from src.config import Configuration


CONFIG = Configuration()
metadata = load_json(CONFIG.videos_data)

for video in metadata[0]:
    print(f"Processing {video['video_path']}...")
    video_path = os.path.join(CONFIG.videos_path, video["video"])

    tone, events = extract_audio_cues(video_path)

    print(f"Tone: {tone}")
    print(f"Background Sounds: {events}")